In [ ]:
import os
import glob
import pickle
import toml

In [ ]:

def parse_ref_file(filepath):
    header = None
    seq_chunks = []

    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_chunks)
                header = line[1:]
                seq_chunks = []
            else:
                seq_chunks.append(line)

        if header is not None:
            yield header, "".join(seq_chunks)


def extract_fields(header):
    parts = header.split("\t")
    sequence_id = parts[0].strip() if len(parts) > 0 else ""
    original_label = parts[1].strip() if len(parts) > 1 else ""
    species = parts[2].strip() if len(parts) > 2 else ""
    return sequence_id, original_label, species


def main(input_dir="."):
    with open(MAPPING_FILE, "r") as f:
        mapping = toml.load(f)

    result = {}
    dropped = 0

    for path in sorted(glob.glob(os.path.join(input_dir, "*.ref"))):
        filename = os.path.basename(path)

        for header, sequence in parse_ref_file(path):
            sequence = sequence.lower()
            sequence_id, original_label, species = extract_fields(header)

            # Terrier special case
            if not original_label and filename == "simple.ref":
                original_label = "Simple Repeat"
            elif not original_label and sequence_id.startswith("SINE_"):
                original_label = "SINE"

            mapped_label = mapping.get(original_label)

            if mapped_label is None or mapped_label == "Unknown":
                dropped += 1
                continue
            if len(sequence) < 80:
                dropped += 1
                continue

            result[sequence] = {
                "sequence_id": sequence_id,
                "original_label": original_label,
                "mapped_label": mapped_label,
                "species": species,
            }

    with open(OUTPUT_PICKLE, "wb") as f:
        pickle.dump(result, f)

    print(f"Saved {len(result)} sequences to {OUTPUT_PICKLE}")
    print(f"Dropped {dropped} unmapped sequences")


In [ ]:
if __name__ == "__main__":
    input_dir = "RepBase31.04.fasta"  # the main repbase input file in the current directory containing .ref files
    MAPPING_FILE = "repeatmasker_labels.toml"
    OUTPUT_PICKLE = "RepBase31pt04_converted2_terrierlabelsMay13.pkl"
    main(input_dir)


Saved 116787 sequences to RepBase31pt04_converted2_terrierlabelsMay13.pkl
Dropped 2687 unmapped sequences


In [ ]:
with open(OUTPUT_PICKLE, "rb") as f:
    data = pickle.load(f)

for seq, info in list(data.items())[:]:
    if len(info['mapped_label'].split("/"))==1:
        print(f"Sequence ID: {info['sequence_id']}")
        print(f"Original Label: {info['original_label']}")
        print(f"Mapped Label: {info['mapped_label']}")
        print(f"Species: {info['species']}")
        print(f"Sequence (first 50 chars): {seq[:50]}...")
        print("-" * 40)

Sequence ID: Clu-137_AG
Original Label: DNA transposon
Mapped Label: DNA
Species: Anopheles gambiae
Sequence (first 50 chars): cccaagtagccttaagaaactctatttgattattaacagttttttaaagc...
----------------------------------------
Sequence ID: DNA-5_AG
Original Label: DNA transposon
Mapped Label: DNA
Species: Anopheles gambiae
Sequence (first 50 chars): tacagtctgttcccgagttacacggttctcgacttacgcggattcggaga...
----------------------------------------
Sequence ID: Clu-166_AG
Original Label: DNA transposon
Mapped Label: DNA
Species: Anopheles gambiae
Sequence (first 50 chars): tacagtcagtccaaaaagtattcgtctagctgaatattttgcagaaaaag...
----------------------------------------
Sequence ID: Clu-11_AG
Original Label: DNA transposon
Mapped Label: DNA
Species: Anopheles gambiae
Sequence (first 50 chars): tagcggaatttggggcaagtgtgccatacggggcaagagtgccaccaagc...
----------------------------------------
Sequence ID: MSAT-3_AG
Original Label: MSAT
Mapped Label: Satellite
Species: Anopheles gambiae
Sequence (first 50 c